# Week 5 - Apache Spark DataFrame Assignment

## Submitted By

**Name:** Aryan Dash

**Course:** B.Tech Computer Science Engineering

**Internship:** Celebal Technologies Data Engineering Internship

---

## Objective

The objective of this assignment is to understand Apache Spark fundamentals and perform data cleaning, transformation, filtering, aggregation, schema modification, and complete data processing using Spark DataFrames. The assignment also demonstrates Spark's in-memory processing, DataFrame immutability, shuffle operations, and aggregation techniques using a sales dataset.

# Dataset Information

**Dataset Name:** sales_data_500.csv

**Number of Records:** 520

**Dataset Columns:**

- user_id
- transaction_date
- store_id
- product_category
- sale_amount
- price
- region
- city
- age
- subscription
- status
- email
- username
- raw_timestamp

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
spark = SparkSession.builder \
    .appName("Week5 Spark Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully!")

C:\Users\91993\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Session Created Successfully!


In [4]:
df = spark.read.csv(
    "sales_data_500.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully!")

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/C:/Users/91993/sales_data_500.csv. SQLSTATE: 42K03

In [5]:
df = spark.read.csv(
    r""C:\Users\91993\OneDrive\Desktop\Week-5_Spark_Assignment\sales_data_500.csv"",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully!")

SyntaxError: '(' was never closed (4062642339.py, line 1)

In [6]:
df = spark.read.csv(
    r"C:\Users\91993\OneDrive\Desktop\Week-5_Spark_Assignment\sales_data_500.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [7]:
df.show(10, truncate=False)

+-------+----------------+--------+----------------+-----------+-------+------+-----------+---+------------+---------+----------------+--------+-------------------+
|user_id|transaction_date|store_id|product_category|sale_amount|price  |region|city       |age|subscription|status   |email           |username|raw_timestamp      |
+-------+----------------+--------+----------------+-----------+-------+------+-----------+---+------------+---------+----------------+--------+-------------------+
|1081   |2024-02-27      |1       |Furniture       |1299.97    |461.64 |North |Lucknow    |23 |Standard    |NULL     |NULL            |user1   |2024-04-29 16:38:00|
|1003   |2024-10-14      |7       |Sports          |2155.65    |1375.17|East  |Delhi      |28 |Standard    |NULL     |user2@gmail.com |user2   |2024-06-21 03:05:00|
|1048   |2024-02-19      |12      |Furniture       |3058.26    |2431.03|West  |Lucknow    |25 |Basic       |Completed|user3@gmail.com |user3   |2024-11-12 11:36:00|
|1024   |2

In [8]:
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)



In [9]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 520
Total Columns: 14


In [10]:
print(df.columns)

['user_id', 'transaction_date', 'store_id', 'product_category', 'sale_amount', 'price', 'region', 'city', 'age', 'subscription', 'status', 'email', 'username', 'raw_timestamp']


# Objective 1: Understanding the Dataset

# Objective 5: Data Cleaning – Remove Duplicate Records

Duplicate records can lead to incorrect analysis and aggregation results. Apache Spark provides the `dropDuplicates()` function to remove duplicate rows from a DataFrame. In this task, duplicate records are identified and removed to improve data quality.

In [11]:
print("Rows Before Removing Duplicates:", df.count())

df = df.dropDuplicates()

print("Rows After Removing Duplicates:", df.count())

Rows Before Removing Duplicates: 520
Rows After Removing Duplicates: 500


# Objective 6: Handle Null and Empty Values

Handling missing and inconsistent values is an important step in data preprocessing. In this task, null values are replaced, and rows containing invalid email addresses or empty usernames are removed.

In [12]:
from pyspark.sql.functions import col

# Fill null values
df = df.na.fill({
    "price": 0,
    "status": "Unknown"
})

# Remove rows with null email or empty username
df = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

df.show(5)

+-------+----------------+--------+----------------+-----------+-------+------+-------+---+------------+---------+-----------------+--------+-------------------+
|user_id|transaction_date|store_id|product_category|sale_amount|  price|region|   city|age|subscription|   status|            email|username|      raw_timestamp|
+-------+----------------+--------+----------------+-----------+-------+------+-------+---+------------+---------+-----------------+--------+-------------------+
|   1072|      2024-05-06|      19|          Sports|     294.73| 291.85|  East|Chennai| 31|    Standard|Cancelled| user19@gmail.com|  user19|2024-12-09 20:19:00|
|   1085|      2024-07-21|      13|          Sports|    1081.52|  95.33| North| Mumbai| 53|       Basic|Cancelled|user317@gmail.com| user317|2024-04-25 19:12:00|
|   1040|      2024-08-06|      11|       Furniture|    1934.08| 500.28|  West| Mumbai| 23|     Premium|Completed| user84@gmail.com|  user84|2024-07-09 04:35:00|
|   1003|      2024-06-12|  

# Objective 7: Filtering Data

Filtering helps extract only the required records from a dataset based on specific conditions.

In [13]:
from pyspark.sql.functions import col

filtered_df = df.filter(
    (col("age") >= 18) &
    (col("age") <= 30) &
    (col("subscription") == "Premium")
)

filtered_df.show()

+-------+----------------+--------+----------------+-----------+-------+------+-----------+---+------------+---------+-----------------+--------+-------------------+
|user_id|transaction_date|store_id|product_category|sale_amount|  price|region|       city|age|subscription|   status|            email|username|      raw_timestamp|
+-------+----------------+--------+----------------+-----------+-------+------+-----------+---+------------+---------+-----------------+--------+-------------------+
|   1040|      2024-08-06|      11|       Furniture|    1934.08| 500.28|  West|     Mumbai| 23|     Premium|Completed| user84@gmail.com|  user84|2024-07-09 04:35:00|
|   1083|      2024-06-27|       4|        Clothing|    4964.95|1209.52| South|  Hyderabad| 18|     Premium|Cancelled|user151@gmail.com| user151|2024-08-05 17:24:00|
|   1064|      2024-11-30|      16|         Grocery|    3455.01|1276.11|  West|     Jaipur| 28|     Premium|Completed|user188@gmail.com| user188|2024-01-17 08:14:00|
|   

# Objective 8: Aggregation Functions

Aggregation functions summarize data by calculating statistics such as count, sum, average, minimum, and maximum values.

In [14]:
from pyspark.sql.functions import count, sum, avg, min, max

df.select(
    count("*").alias("Total Records"),
    sum("price").alias("Total Price"),
    avg("price").alias("Average Price"),
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price")
).show()

+-------------+-----------------+------------------+-------------+-------------+
|Total Records|      Total Price|     Average Price|Minimum Price|Maximum Price|
+-------------+-----------------+------------------+-------------+-------------+
|          452|697031.8399999995|1542.1058407079636|          0.0|      2997.88|
+-------------+-----------------+------------------+-------------+-------------+



# Objective 9: GroupBy and Aggregation

The `groupBy()` operation groups similar records together and applies aggregation functions to each group.

In [15]:
from pyspark.sql.functions import avg

df.filter(col("region") == "West") \
.groupBy("product_category") \
.agg(avg("sale_amount").alias("Average Sale")) \
.show()

+----------------+------------------+
|product_category|      Average Sale|
+----------------+------------------+
|          Sports|2699.4860869565214|
|         Grocery| 3422.475217391305|
|     Electronics|1891.4819047619046|
|        Clothing|2558.3705263157885|
|       Furniture|2934.1052173913044|
+----------------+------------------+



# Objective 10: Wide Transformations and Shuffle

Operations such as `groupBy()` require data to be redistributed across partitions. This process is called a **shuffle**. Since data moves between partitions, these operations are known as **wide transformations**.

Examples include:
- groupBy()
- join()
- distinct()
- reduceByKey()

# Objective 11: Schema Modification

Schema modification includes changing data types and renaming columns to improve readability and compatibility.

In [16]:
from pyspark.sql.types import TimestampType

df = df.withColumn(
    "event_time",
    col("raw_timestamp").cast(TimestampType())
).drop("raw_timestamp")

df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = false)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = false)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



# Objective 12: Complete Data Processing Pipeline

This pipeline combines duplicate removal, null handling, grouping, aggregation, and sorting into one workflow.

In [17]:
from pyspark.sql.functions import desc, sum

result = (
    df.dropDuplicates()
      .na.fill({"price": 0})
      .groupBy("store_id")
      .agg(sum("price").alias("Total_Revenue"))
      .orderBy(desc("Total_Revenue"))
)

result.show()

+--------+------------------+
|store_id|     Total_Revenue|
+--------+------------------+
|       8|          60106.62|
|      16|          48272.36|
|       6|44453.670000000006|
|      10|          44239.63|
|      18|          43278.47|
|      17| 41160.50999999999|
|      11| 40355.46000000001|
|      20|38094.670000000006|
|       9|35713.530000000006|
|      19|          31029.14|
|      14|          30735.01|
|       5|29877.799999999996|
|      13|29839.199999999997|
|      12|28835.630000000005|
|       3|28760.239999999998|
|       4|28183.390000000003|
|       2|          27935.33|
|       1|26914.790000000005|
|      15|23832.340000000004|
|       7|15414.050000000001|
+--------+------------------+



# Objective 13: Query Results

The outputs generated above demonstrate successful execution of data cleaning, filtering, aggregation, grouping, schema modification, and the complete processing pipeline using PySpark DataFrames.

# Objective 14: Overall Insights

- Spark performs data processing efficiently using in-memory computation.
- DataFrames are immutable; each transformation creates a new DataFrame.
- Data cleaning improves data quality and accuracy.
- Filtering extracts relevant records for analysis.
- Aggregation functions summarize the dataset effectively.
- GroupBy operations involve shuffle and are classified as wide transformations.
- Schema modification ensures correct data types.
- A complete processing pipeline integrates all transformation steps into a single workflow.

In [1]:
# Week-5 Assignment Questions (Q1–Q15)

## Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

### Answer

Traditional MapReduce has several limitations that make Apache Spark a better choice for modern big data processing.

**Limitations of MapReduce:**
1. It stores intermediate results on disk, making processing slower.
2. It requires multiple MapReduce jobs for iterative tasks.
3. It is not suitable for real-time data processing.
4. It has higher disk I/O, increasing execution time.
5. Writing MapReduce programs is more complex.

**Advantages of Spark:**
1. Spark performs in-memory computing, which is much faster.
2. It supports batch processing, streaming, machine learning, and graph processing.
3. Spark executes iterative algorithms efficiently.
4. It provides high-level APIs in Python, Java, Scala, and R.
5. It offers better fault tolerance and faster data processing than MapReduce.

**Conclusion:**  
Because Spark processes data in memory instead of repeatedly reading and writing to disk, it is significantly faster and more efficient for modern big data applications.

## Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.


### Answer

Apache Spark uses **in-memory computing**, which means it stores intermediate data in RAM instead of writing it to disk after every operation. This significantly reduces disk input/output (I/O) and speeds up data processing.

In iterative machine learning algorithms, the same dataset is processed multiple times. Spark keeps the data in memory, allowing repeated operations to access it quickly. In contrast, traditional disk-based systems such as MapReduce read and write data to disk after every iteration, making the process much slower.

**Advantages of In-Memory Computing in Spark:**

1. Faster execution because data is stored in RAM.
2. Reduced disk I/O operations.
3. Better performance for iterative machine learning and graph algorithms.
4. Lower execution time for repeated computations.
5. Improved overall efficiency for big data processing.

**Conclusion:**

Spark's in-memory computing makes iterative machine learning algorithms much faster than traditional disk-based systems by minimizing disk access and maximizing processing speed.

## Q3. Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: `user_id` and `transaction_date`.

In [2]:
# Remove duplicate rows based on user_id and transaction_date
df_q3 = df.dropDuplicates(["user_id", "transaction_date"])

print("Number of records after removing duplicates:")
print(df_q3.count())

# Display first 10 records
df_q3.show(10)

NameError: name 'df' is not defined

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [4]:
spark = SparkSession.builder \
    .appName("Week5 Spark Assignment") \
    .getOrCreate()

C:\Users\91993\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [5]:
df = spark.read.csv(
    r"C:\Users\91993\OneDrive\Desktop\Week-5_Spark_Assignment\sales_data_500.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+-------+----------------+--------+----------------+-----------+-------+------+---------+---+------------+---------+---------------+--------+-------------------+
|user_id|transaction_date|store_id|product_category|sale_amount|  price|region|     city|age|subscription|   status|          email|username|      raw_timestamp|
+-------+----------------+--------+----------------+-----------+-------+------+---------+---+------------+---------+---------------+--------+-------------------+
|   1081|      2024-02-27|       1|       Furniture|    1299.97| 461.64| North|  Lucknow| 23|    Standard|     NULL|           NULL|   user1|2024-04-29 16:38:00|
|   1003|      2024-10-14|       7|          Sports|    2155.65|1375.17|  East|    Delhi| 28|    Standard|     NULL|user2@gmail.com|   user2|2024-06-21 03:05:00|
|   1048|      2024-02-19|      12|       Furniture|    3058.26|2431.03|  West|  Lucknow| 25|       Basic|Completed|user3@gmail.com|   user3|2024-11-12 11:36:00|
|   1024|      2024-12-26|  

## Q3. Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: `user_id` and `transaction_date`.

In [7]:
df_q3 = df.dropDuplicates(["user_id", "transaction_date"])

print("Number of records after removing duplicates:")
print(df_q3.count())

df_q3.show(10)

Number of records after removing duplicates:
495
+-------+----------------+--------+----------------+-----------+-------+------+---------+---+------------+---------+-----------------+--------+-------------------+
|user_id|transaction_date|store_id|product_category|sale_amount|  price|region|     city|age|subscription|   status|            email|username|      raw_timestamp|
+-------+----------------+--------+----------------+-----------+-------+------+---------+---+------------+---------+-----------------+--------+-------------------+
|   1000|      2024-04-01|       3|          Sports|    4445.88|1277.86|  West|Bengaluru| 31|     Premium|Cancelled|user433@gmail.com| user433|2024-03-30 19:44:00|
|   1000|      2024-09-11|      14|        Clothing|     733.01|1621.12| South|  Lucknow| 53|    Standard|Cancelled| user91@gmail.com|  user91|2024-01-22 13:01:00|
|   1000|      2024-09-18|      12|          Sports|    3271.98| 1846.7|  East|   Mumbai| 48|     Premium|Cancelled|user250@gmail.c

# Q4. Filter records where region = 'West' and find the average sale_amount for each product_category.

In [8]:
from pyspark.sql.functions import avg

# Filter rows where region is West
df_q4 = df.filter(df.region == "West")

# Group by product_category and calculate average sale_amount
result_q4 = df_q4.groupBy("product_category") \
                 .agg(avg("sale_amount").alias("Average_Sale_Amount"))

print("Average Sale Amount in West Region:")
result_q4.show()

Average Sale Amount in West Region:
+----------------+-------------------+
|product_category|Average_Sale_Amount|
+----------------+-------------------+
|          Sports| 2682.7739999999994|
|         Grocery| 3468.7510714285722|
|     Electronics| 1841.1346153846157|
|        Clothing| 2570.6428571428573|
|       Furniture| 2765.1314285714284|
+----------------+-------------------+



# Q5. Difference between .na.drop() and .na.fill()

### Answer:

- **.na.drop()**
  - Removes rows that contain null values.
  - Useful when incomplete records should not be included in the analysis.

- **.na.fill()**
  - Replaces null values with a specified value instead of removing the rows.
  - Useful when you want to keep all records while handling missing values.

In this example, null values in the **status** column are replaced with **"Unknown"**.

# Q6. Find the total count of records for each city where the count is greater than 100.

In [9]:
from pyspark.sql.functions import count

# Count records for each city
df_q6 = df.groupBy("city") \
          .agg(count("*").alias("Total_Records")) \
          .filter(col("Total_Records") > 100)

print("Cities having more than 100 records:")
df_q6.show()

Cities having more than 100 records:
+----+-------------+
|city|Total_Records|
+----+-------------+
+----+-------------+



# Q7. How does the immutability of Spark DataFrames affect how you perform data cleaning steps like dropping columns or renaming them?

### Answer

Spark DataFrames are **immutable**, which means they **cannot be modified directly** after they are created. Instead of changing the original DataFrame, every transformation creates a **new DataFrame**.

For example, when dropping a column or renaming a column, Spark returns a new DataFrame containing the changes, while the original DataFrame remains unchanged.

This approach provides several benefits:

- Prevents accidental modification of the original data.
- Makes data processing more reliable and fault-tolerant.
- Allows transformations to be chained together efficiently.
- Supports Spark's optimization engine (Catalyst Optimizer) for better performance.

**Example:**

```python
new_df = df.drop("status")
new_df = df.withColumnRenamed("price", "product_price")
```

In the above example, the original `df` is unchanged. The modified DataFrames are stored in new variables.


# Q8. Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [10]:
from pyspark.sql.functions import col

# Filter users with age between 18 and 30 and Premium subscription
df_q8 = df.filter(
    (col("age") >= 18) &
    (col("age") <= 30) &
    (col("subscription") == "Premium")
)

print("Filtered Records:")
df_q8.show()

Filtered Records:
+-------+----------------+--------+----------------+-----------+-------+------+-----------+---+------------+---------+-----------------+--------+-------------------+
|user_id|transaction_date|store_id|product_category|sale_amount|  price|region|       city|age|subscription|   status|            email|username|      raw_timestamp|
+-------+----------------+--------+----------------+-----------+-------+------+-----------+---+------------+---------+-----------------+--------+-------------------+
|   1059|      2024-01-28|       4|     Electronics|    2072.84|1050.94| North|  Hyderabad| 30|     Premium|     NULL| user15@gmail.com|  user15|2024-08-24 07:55:00|
|   1052|      2024-11-13|       5|        Clothing|    4336.57|2929.81| South|       Pune| 19|     Premium|Cancelled| user25@gmail.com|  user25|2024-12-08 23:51:00|
|   1056|      2024-02-09|      10|        Clothing|     1431.4| 993.14| North|  Bengaluru| 27|     Premium|     NULL| user36@gmail.com|  user36|2024-02

# Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like `sum()` or `avg()`?


### Answer

It is important to handle null values before performing mathematical aggregations such as `sum()` or `avg()` because missing values can affect the accuracy and reliability of the results.

**Reasons:**

1. Null values may produce incorrect or incomplete aggregation results.
2. Replacing or removing null values improves data quality.
3. It ensures that statistical calculations are more accurate.
4. Clean data leads to more reliable analysis and better decision-making.
5. Handling null values before aggregation prevents unexpected errors during data processing.

**Example:**

If the `price` column contains null values, they can be replaced with **0** before calculating the total or average.

```python
df = df.na.fill({"price": 0})
```

**Conclusion:**

Handling null values before performing mathematical operations ensures that aggregation functions produce accurate and meaningful results.

# Q10. Write the code to revise a column named `raw_timestamp` by casting it to a `TimestampType` and renaming it to `event_time`.

In [11]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

# Cast raw_timestamp to TimestampType and rename it to event_time
df_q10 = df.withColumn(
    "event_time",
    col("raw_timestamp").cast(TimestampType())
).drop("raw_timestamp")

print("Updated Schema:")
df_q10.printSchema()

print("Sample Data:")
df_q10.select("event_time").show(10, truncate=False)

Updated Schema:
root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

Sample Data:
+-------------------+
|event_time         |
+-------------------+
|2024-04-29 16:38:00|
|2024-06-21 03:05:00|
|2024-11-12 11:36:00|
|2024-03-24 11:22:00|
|2024-10-12 07:43:00|
|2024-11-25 14:09:00|
|2024-09-09 02:48:00|
|2024-01-06 21:46:00|
|2024-05-14 16:48:00|
|2024-02-27 11:56:00|
+-------------------+
only showing top 10 rows


## Q11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?



### Answer

A **shuffle** is the process in Apache Spark where data is **redistributed across different partitions** during certain operations such as `groupBy()`, `join()`, `distinct()`, and `reduceByKey()`.

During a `groupBy()` operation, Spark moves records with the same key to the same partition so that aggregation functions such as `count()`, `sum()`, and `avg()` can be performed correctly.

It is called a **wide transformation** because data is transferred between multiple partitions across the cluster. Unlike narrow transformations, wide transformations require communication between partitions, which makes them more expensive in terms of execution time.

### Key Points

- Shuffle redistributes data across partitions.
- It occurs during operations like `groupBy()`, `join()`, and `distinct()`.
- It increases network and disk I/O.
- It is considered a **wide transformation** because data moves between partitions.
- Minimizing unnecessary shuffles improves Spark performance.

**Conclusion:**

Shuffle is an important Spark process that enables grouping and aggregation operations. Although it increases processing time due to data movement, it is necessary for correct distributed computation.

## Q12. Write a code snippet that identifies and removes rows where the `email` column contains null values OR the `username` is an empty string.

In [12]:
from pyspark.sql.functions import col

# Identify rows with null email or empty username
invalid_rows = df.filter(
    col("email").isNull() | (col("username") == "")
)

print("Invalid Records:")
invalid_rows.show()

# Remove invalid rows
df_q12 = df.filter(
    col("email").isNotNull() & (col("username") != "")
)

print("Cleaned Dataset:")
df_q12.show(10)

print("Total Records After Cleaning:", df_q12.count())

Invalid Records:
+-------+----------------+--------+----------------+-----------+-------+------+-----------+---+------------+---------+-----+--------+-------------------+
|user_id|transaction_date|store_id|product_category|sale_amount|  price|region|       city|age|subscription|   status|email|username|      raw_timestamp|
+-------+----------------+--------+----------------+-----------+-------+------+-----------+---+------------+---------+-----+--------+-------------------+
|   1081|      2024-02-27|       1|       Furniture|    1299.97| 461.64| North|    Lucknow| 23|    Standard|     NULL| NULL|   user1|2024-04-29 16:38:00|
|   1036|      2024-09-20|      20|       Furniture|      557.6| 2858.4| South|    Chennai| 32|     Premium|  Pending| NULL|  user32|2024-08-31 19:54:00|
|   1083|      2024-03-18|      16|     Electronics|     718.61|2897.79|  East|     Mumbai| 46|     Premium|     NULL| NULL|  user64|2024-12-01 04:04:00|
|   1043|      2024-02-28|      15|     Electronics|     78

## Q13. How do you use the `.agg()` function to calculate multiple statistics at once, such as the minimum, maximum, and mean of the `price` column?

In [13]:
from pyspark.sql.functions import min, max, avg

# Calculate multiple statistics using agg()
df_q13 = df.agg(
    min("price").alias("Minimum_Price"),
    max("price").alias("Maximum_Price"),
    avg("price").alias("Average_Price")
)

print("Statistics of Price Column:")
df_q13.show()

Statistics of Price Column:
+-------------+-------------+------------------+
|Minimum_Price|Maximum_Price|     Average_Price|
+-------------+-------------+------------------+
|         63.9|      2997.88|1599.2221471172945|
+-------------+-------------+------------------+



## Q14. In the context of cleaning a dataset, what is the risk of using `inferSchema=True` when your source data contains messy or inconsistent date formats?

### Answer

The `inferSchema=True` option automatically detects the data type of each column while reading the dataset. However, if the dataset contains messy or inconsistent date formats, Spark may infer an incorrect data type.

### Risks of using `inferSchema=True`

1. Incorrect date formats may be read as **StringType** instead of **DateType** or **TimestampType**.
2. Different date formats in the same column can cause parsing errors.
3. Date-based operations such as filtering, sorting, and aggregation may produce incorrect results.
4. Additional data cleaning and type conversion may be required.
5. Schema inference may vary depending on the sample of data read.

### Conclusion

When datasets contain inconsistent date formats, it is better to define the schema manually or clean the date values before performing analysis. This ensures accurate and reliable data processing.

## Q15. Write a final processing pipeline that:

- Filters out duplicates.
- Fills null prices with 0.
- Groups by `store_id`.
- Calculates total revenue.

In [14]:
from pyspark.sql.functions import sum

# Final Data Processing Pipeline
df_q15 = (
    df.dropDuplicates()
      .na.fill({"price": 0})
      .groupBy("store_id")
      .agg(sum("price").alias("Total_Revenue"))
      .orderBy("store_id")
)

print("Final Processing Pipeline Result:")
df_q15.show()

Final Processing Pipeline Result:
+--------+------------------+
|store_id|     Total_Revenue|
+--------+------------------+
|       1| 32715.78000000001|
|       2|          33917.14|
|       3| 28760.23999999999|
|       4|32298.370000000003|
|       5|           29877.8|
|       6| 46789.11000000001|
|       7|          20881.03|
|       8| 60447.02999999999|
|       9|          38448.28|
|      10| 49726.74999999999|
|      11|          44824.66|
|      12|30952.280000000002|
|      13|          35564.98|
|      14|          35903.45|
|      15|32476.829999999998|
|      16|          52055.38|
|      17|          41160.51|
|      18|          44663.62|
|      19|35496.880000000005|
|      20|          45621.06|
+--------+------------------+



# Conclusion

In this assignment, Apache Spark DataFrame operations were successfully performed to clean, transform, filter, and analyze a sales dataset. Various Spark concepts such as DataFrame immutability, in-memory computing, schema modification, shuffle operations, filtering, grouping, and aggregation were implemented using PySpark.

The assignment demonstrated how Spark efficiently processes large datasets through distributed computing and optimized DataFrame transformations. All the objectives and Week-5 questions were completed successfully, providing practical experience in data preprocessing and analysis using Apache Spark.